In [25]:
import numpy as np
import subprocess
import sys
import os
import pickle
import shutil

In [26]:
ruta_raiz = 'C:/Users/Usuario/Documents/GitHub/RRAM_Simulation/'
# ruta_raiz = '/Users/antonio_lopez_torres/Documents/GitHub/RRAM_Simulation/'  # Ruta en el mac
sys.path.append(ruta_raiz)

# Ruta proporcionada
ruta_exp_data = ruta_raiz + 'Datos_Experimentales/Ciclos_Experimentales/'
ruta_init_data = ruta_raiz + 'Initial_data/'
ruta_init_state_reset = ruta_raiz + 'Estados_antes_reset/'

# Borro los datos iniciales de la simulación anterior verifica si la carpeta existe
if os.path.exists(ruta_init_data):
    # Elimina la carpeta y su contenido
    shutil.rmtree(ruta_init_data)
    # crea la carpeta
    os.makedirs(ruta_init_data)
else:
    # crea la carpeta
    os.makedirs(ruta_init_data)


 # Borro los datos iniciales de la simulación anterior verifica si la carpeta existe
if os.path.exists(ruta_init_state_reset):
    # Elimina la carpeta y su contenido
    shutil.rmtree(ruta_init_state_reset)
    # crea la carpeta
    os.makedirs(ruta_init_state_reset)
else:
    # crea la carpeta
    os.makedirs(ruta_init_state_reset)

In [27]:
# Solicitar la ruta del archivo al usuario
ruta_archivo_set = ruta_exp_data + '/Cycle_p_1000.txt'
ruta_archivo_reset = ruta_exp_data + '/Cycle_n_1000.txt'

# Leer datos del archivo
data_set = np.loadtxt(ruta_archivo_set)
data_reset = np.loadtxt(ruta_archivo_reset)

ruta_init_script = ruta_raiz + 'Init_simulation.py'
ruta_simulation_script = ruta_raiz + 'RRAM_Simulation_exceptions.py'

# Defino la carpeta donde se guardan los datos iniciales de la simulación
carpeta_results = 'Results'

simulation_path = os.path.join(carpeta_results)

# Verifica si la carpeta existe
if os.path.exists(simulation_path):
    # Elimina la carpeta y su contenido
    shutil.rmtree(simulation_path)

    # crea la carpeta
    os.makedirs(simulation_path)
    os.makedirs(simulation_path + '/Figures')

In [28]:
# Creo los vectores y luego los combino para generar los sets de datos a probar
# Creo los vectores y luego los combino para generar los sets de datos a probar
I_0 = [2e-3]
ohm_resistence = [1.0,1.25,1.5, 1.75, 2.00, 2.25, 2.5, 2.75, 3.0] # 1.5 es el valor óptimo
gamma = [8,8,9,9] #10, 11, 12]
E_a = [1.05]
factor_generacion = [1.2] # 1,1 1,2 funcion bn
r_termica_no_percola = [10]
r_termica_percola = [500] #, 9e2, 7e2]

In [29]:
# Generamos la lista de tuplas con la combinación de valores
data = [(v1, v2, v3, v4, v5, v6, v7) for v2 in gamma for v1 in ohm_resistence for v3 in I_0 for v4 in E_a for v5 in factor_generacion for v6 in r_termica_no_percola for v7 in r_termica_percola]

# print("data:", data)

# Extraemos los valores usando zip
ohm_resistence, gamma, I_0, E_a, factor_generacion, r_termica_no_percola, r_termica_percola = zip(*data)

# Convertimos a listas nuevamente
I_0 = list(I_0)
ohm_resistence = list(ohm_resistence)
gamma = list(gamma)
E_a = list(E_a)
factor_generacion = list(factor_generacion)
r_termica_no_percola = list(r_termica_no_percola)
r_termica_percola = list(r_termica_percola)

# print("ohm_resistence:", ohm_resistence)
# print("gamma:", gamma)
# print("I_0:", I_0)
# print("E_a:", E_a)
# print("factor_generacion:", factor_generacion)
# print("r_termica_no_percola:", r_termica_no_percola)
# print("r_termica_percola:", r_termica_percola)

In [30]:
# Guardo los datos en un archivo pkl
with open(ruta_init_data + 'I_0.pkl', 'wb') as f:
    pickle.dump(I_0, f)

with open(ruta_init_data + 'ohm_resistence.pkl', 'wb') as f:
    pickle.dump(ohm_resistence, f)

with open(ruta_init_data + 'gamma.pkl', 'wb') as f:
    pickle.dump(gamma, f)

with open(ruta_init_data + 'E_a.pkl', 'wb') as f:
    pickle.dump(E_a, f)
    
with open(ruta_init_data + 'factor_generacion.pkl', 'wb') as f:
    pickle.dump(factor_generacion, f)
    
with open(ruta_init_data + 'r_termica_no_percola.pkl', 'wb') as f:
    pickle.dump(r_termica_no_percola, f)
    
with open(ruta_init_data + 'r_termica_percola.pkl', 'wb') as f:
    pickle.dump(r_termica_percola, f)

In [31]:
# LLamo al script de configuracion que genera los archivos de configuracion
subprocess.run(["python", ruta_init_script, ruta_init_data, str(len(ohm_resistence))])

CompletedProcess(args=['python', 'C:/Users/Usuario/Documents/GitHub/RRAM_Simulation/Init_simulation.py', 'C:/Users/Usuario/Documents/GitHub/RRAM_Simulation/Initial_data/', '36'], returncode=0)

In [ ]:
import subprocess
import concurrent.futures
import os

# Carpeta donde se guardarán los logs
log_dir = "logs"
os.makedirs(log_dir, exist_ok=True)  # Crea la carpeta si no existe

# Limpio la carpeta de logs antes de empezar
for file in os.listdir(log_dir):
    file_path = os.path.join(log_dir, file)
    try:
        if os.path.isfile(file_path):
            os.unlink(file_path)
    except Exception as e:
        print(f"No se pudo eliminar {file_path}. Error: {e}")
# Función que ejecuta una simulación y guarda su propio log

guardar_datos = True

def ejecutar_simulacion(num_simulacion):
    log_file_path = os.path.join(log_dir, f"log_simulacion_{num_simulacion + 1}.log")

    with open(log_file_path, 'w') as log_file:
        print(f"Iniciando simulación {num_simulacion+1}")
        subprocess.run(["python", 'C:/Users/Usuario/Documents/GitHub/RRAM_Simulation/RRAM_Simulation_exceptions.py', str(num_simulacion), str(guardar_datos)],
                       stdout=log_file, stderr=log_file)


# Paralelización con hilos (funciona en Jupyter)
num_simulaciones = len(ohm_resistence)

print(f"Se van a ejecutar {num_simulaciones} simulaciones:\n")

with concurrent.futures.ThreadPoolExecutor(max_workers=8) as executor:  # Usa hilos en vez de procesos
    executor.map(ejecutar_simulacion, range(num_simulaciones))

print("Todas las simulaciones han terminado")

Se van a ejecutar 36 simulaciones:

Iniciando simulación 1
Iniciando simulación 2
Iniciando simulación 3
Iniciando simulación 4
Iniciando simulación 5
Iniciando simulación 6
Iniciando simulación 7
Iniciando simulación 8
Iniciando simulación 9
Iniciando simulación 10
Iniciando simulación 11
Iniciando simulación 12
Iniciando simulación 13
Iniciando simulación 14
Iniciando simulación 15
Iniciando simulación 16
Iniciando simulación 17
Iniciando simulación 18
Iniciando simulación 19
Iniciando simulación 20
Iniciando simulación 21
Iniciando simulación 22
Iniciando simulación 23
Iniciando simulación 24
Iniciando simulación 25
Iniciando simulación 26
Iniciando simulación 27
Iniciando simulación 28
Iniciando simulación 29
Iniciando simulación 30
Iniciando simulación 31
Iniciando simulación 32
Iniciando simulación 33
Iniciando simulación 34
Iniciando simulación 35
Iniciando simulación 36
